In [ ]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector

In [ ]:
def column_ratio(X):
    return X[:, [0]] / (X[:, [1]] + 0.1**10)

def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]

def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler())

def app_temp(X):
    t = X[:, [0]]
    h = X[:, [1]]
    return t -  (t - 10) * (0.8 - h/100) / 2.3

def app_temp_name(function_transformer, feature_names_in):
    return ["temp"]

def app_temp_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(app_temp, feature_names_out=app_temp_name),
        StandardScaler())

num_pipeline = make_pipeline(SimpleImputer(strategy="median"),
                                     StandardScaler())

cat_pipeline = OneHotEncoder()

preprocessing = ColumnTransformer([
    ("app_temp", app_temp_pipeline(), ["Temparature", "Humidity"]),
    ("nit_pot_ratio", ratio_pipeline(), ["Nitrogen", "Potassium"]),
    ("nit_pho_ratio", ratio_pipeline(), ["Nitrogen", "Phosphorous"]),
    ("pot_pho_ratio", ratio_pipeline(), ["Potassium", "Phosphorous"]),
    ("num", num_pipeline, make_column_selector(dtype_include=np.number)),
    ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
])


元の訓練データから "id" と "Fertilizer Name" を除いた X を渡す

In [ ]:
X_prepared = preprocessing.fit_transform(X)

X_prepared は numpy配列 なので DataFrame形式 に直したいときは以下のコードを実行する

In [ ]:
X_prepared_df = pd.DataFrame(X_prepared,
                             columns=preprocessing.get_feature_names_out(),
                             index=X.index)